# VersionedClass Tutorial

## Introduction

The `VersionedClass` is the core building block of the `classversioning` package. It enables the creation of class hierarchies where each subclass is associated with a specific version.

This tutorial focuses specifically on `VersionedClass` capabilities, including:
- Creating a versioned class hierarchy.
- Automatic registration of subclasses.
- Retrieving classes based on version.
- Dispatching object construction to the correct versioned subclass.


## Importing the Module

We need `VersionedClass`, `VersionRegistry`, and a version implementation like `TriNumberVersion`.


In [1]:
from typing import Any
from classversioning import VersionedClass, TriNumberVersion, VersionRegistry


## Core Functionality

### Defining a Head Class

The root of a versioned hierarchy is called the "head class". It must inherit from `VersionedClass` and define the `VERSION_TYPE`.
We also specify `class_registry_type = VersionRegistry` to ensure the correct registry type is used for storing versions, and `class_registration = True` to enable the registry.


In [2]:
class DataProcessor(VersionedClass):
    """Head class for data processors."""
    class_registration = True
    class_registry_type = VersionRegistry
    VERSION_TYPE = TriNumberVersion


### Defining Versioned Subclasses

Subclasses inherit from the head class (or other subclasses) and specify their `VERSION`.


In [3]:
class DataProcessorV1(DataProcessor):
    """Data processor for version 1.0.0."""
    VERSION = TriNumberVersion(1, 0, 0)

    def process(self) -> str:
        """Processes data with V1 algorithm."""
        return "Processing with V1 algorithm"


class DataProcessorV2(DataProcessor):
    """Data processor for version 2.0.0."""
    VERSION = TriNumberVersion(2, 0, 0)

    def process(self) -> str:
        """Processes data with V2 algorithm."""
        return "Processing with V2 algorithm"


### Retrieving Classes

You can retrieve specific versions directly from the head class using `get_registered_class`.


In [4]:
v1_class = DataProcessor.get_registered_class(TriNumberVersion(1, 0, 0), exact=True)
print(f"Retrieved: {v1_class.__name__}")

# Approximate matching (get latest version <= requested)
v2_request = TriNumberVersion(2, 1, 0)
v2_class = DataProcessor.get_registered_class(v2_request, exact=False)
print(f"Requested {v2_request}, got {v2_class.__name__} ({v2_class.VERSION})")


Retrieved: DataProcessorV1
Requested 2.1.0, got DataProcessorV2 (2.0.0)


## Module Interaction

`VersionedClass` relies on `VersionRegistry` to manage the classes. You can access the underlying registry via the `class_registry` attribute.


In [5]:
print(f"Registry: {DataProcessor.class_registry}")
# Inspecting registered versions in the default group
registry_data = DataProcessor.class_registry.data["default"]
versions = [str(cls.VERSION) for cls in registry_data]
print(f"Registered versions: {versions}")


Registry: {'default': [<class '__main__.DataProcessor'>, <class '__main__.DataProcessorV1'>, <class '__main__.DataProcessorV2'>]}
Registered versions: ['0.0.0', '1.0.0', '2.0.0']


## Advanced Features

### Object-Based Dispatch

`VersionedClass` can automatically dispatch to the correct subclass based on an input object. To enable this, the head class must implement `get_version_from_object`.


In [6]:
class Message(VersionedClass):
    """Head class for messages."""
    class_registration = True
    class_registry_type = VersionRegistry
    VERSION_TYPE = TriNumberVersion

    @classmethod
    def get_version_from_object(cls, obj: dict[str, Any]) -> TriNumberVersion:
        """Extracts version from a message dictionary.

        Args:
            obj: The message dictionary.

        Returns:
            The version object.
        """
        # Assume obj is a dict with a 'version' key
        ver_str = obj.get("version", "0.0.0")
        parts = map(int, ver_str.split("."))
        return TriNumberVersion(*parts)


class MessageV1(Message):
    """Message for version 1.0.0."""
    VERSION = TriNumberVersion(1, 0, 0)


class MessageV2(Message):
    """Message for version 2.0.0."""
    VERSION = TriNumberVersion(2, 0, 0)


# Dispatching
data = {"version": "2.0.0", "content": "hello"}
# Provide the object as the first argument (or via _dispatch_kwarg)
msg_instance = Message(data)
print(f"Created instance of: {type(msg_instance).__name__}")


Created instance of: MessageV2


## API Highlights

- **`VersionedClass`**: Base class.
  - `VERSION_TYPE`: Class attribute specifying the version class (e.g., `TriNumberVersion`).
  - `VERSION`: Class attribute specifying the specific version of a subclass.
  - `get_registered_class(version, exact=False, ...)`: Retrieve a class.
  - `get_latest_version_class()`: Retrieve the latest class.
- **`DispatchableClass`**: `VersionedClass` inherits from this, enabling `__new__` dispatch logic.


## Troubleshooting / FAQs

- **Error: `TypeError: ... missing ... required positional argument`**: When using dispatch, ensure `get_version_from_object` is implemented correctly and handles the arguments passed to the constructor.
- **Version Mismatch**: If `exact=True` is used and the version doesn't exist, a lookup error or `None` might occur (depending on configuration, but typically raises `ValueError` or `KeyError` in registry).


## Conclusion

`VersionedClass` simplifies managing multiple versions of logic or data structures, allowing for clean separation of concerns and easy upgrades.

For more information on the registry, see the `versionregistry_tutorial` (if available) or the package tutorial.
